# SQL with DuckDB: Connecting to a Database

**Lessons 7.4 & 7.5.** DuckDB speaks one SQL dialect over *everything* — local files (CSV, Parquet,
Excel, Stata) and remote **databases**. In this exercise you connect DuckDB to a live PostgreSQL
database holding a small sample of the South Africa Census 2022 (Western Cape), run some basic SQL,
and finish by exporting a result to a file and querying that file directly.

Every query follows the same shape:

```sql
SELECT   columns
FROM     census.table          -- a table in the attached database
WHERE    condition             -- filter   (optional)
GROUP BY column                -- group    (optional)
ORDER BY column DESC           -- sort     (optional)
```

**Tables** (linked by the household id `QID`): `persons`, `households`, `geography`.

⚠️ The values are stored as **text**, so numeric columns like `DERH_HSIZE` and `P04_AGE` need
`TRY_CAST(col AS DOUBLE)` before you can average them.

In [ ]:
import sys, subprocess
from pathlib import Path

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))
from src.utilities.project_paths import DATA_DIR

EXPORT_DIR = DATA_DIR / '01_interim' / 'pg_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Read-only connection string to the shared teaching database
db_url = ("postgresql://postgres:rzBk9HgkFxZhcyhQugfAXJGc26q6HiemoWpseXAd8LoWF2kve1zgK8cyLvspc649"
          "@46.224.178.205:25432/postgres?sslmode=require")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false;")          # keep notebook output tidy
con.execute("INSTALL postgres; LOAD postgres;")          # DuckDB's PostgreSQL connector
con.execute(f"ATTACH '{db_url}' AS census (TYPE postgres, READ_ONLY);")

print('duckdb', duckdb.__version__, '- attached the database as \"census\" (read-only)')

---
# Part A — Connect and Look Around (7.5)

The `ATTACH ... (TYPE postgres, READ_ONLY)` in the setup opened the database as `census`. Its
tables are now addressed as `census.<table>` — exactly like any other SQL table. `READ_ONLY` means
we can query but never modify the server.

## A1. What tables are in the database?

List the tables, then count the rows in each.

In [ ]:
# TODO: list the tables in the 'census' catalog (information_schema.tables,
#       table_catalog = 'census' AND table_schema = 'public')
display(con.sql("""
    SELECT ...
    FROM   information_schema.tables
    WHERE  ...
""").df())

# TODO: count rows in each of persons / households / geography (UNION ALL of three COUNT(*))
con.sql("""
    SELECT 'persons' AS table_name, COUNT(*) AS n_rows FROM census.persons
    ...
""").df()

## A2. Peek at the households table

`SELECT *` with a small `LIMIT` is the quickest way to see what you are working with.

In [ ]:
# TODO: select all columns from census.households, limit to 5 rows
con.sql("""
    SELECT ...
    FROM   census.households
    LIMIT ...
""").df()

## A3. Select specific columns

From `geography`, show `QID`, `District`, `Municipality`, and `Geo_type` for the first 10 rows.

In [ ]:
# TODO: select QID, District, Municipality, Geo_type from census.geography, limit 10
con.sql("""
    SELECT ..., ..., ..., ...
    FROM   census.geography
    LIMIT 10
""").df()

---
# Part B — Filtering with WHERE

`WHERE` keeps only the rows that match a condition. Because the columns are text, comparisons on
labels use quotes (`Geo_type = 'Urban area'`), and numeric comparisons need a cast
(`TRY_CAST(DERH_HSIZE AS INTEGER) >= 5`).

## B1. Count all households

In [ ]:
# TODO: COUNT(*) AS n_households from census.households
con.sql("""
    SELECT ...
    FROM   census.households
""").df()

## B2. Urban households only

In `geography`, `Geo_type` labels each household's settlement type. Count the ones in
`'Urban area'`.

In [ ]:
# TODO: count rows in census.geography WHERE Geo_type = 'Urban area'
con.sql("""
    SELECT COUNT(*) AS n_urban
    FROM   census.geography
    WHERE  ...
""").df()

## B3. Combine conditions

Show female-headed households with 5 or more members: `DERH_HHSEX = 'Female'`
**and** `TRY_CAST(DERH_HSIZE AS INTEGER) >= 5`.

In [ ]:
# TODO: WHERE DERH_HHSEX = 'Female' AND TRY_CAST(DERH_HSIZE AS INTEGER) >= 5
con.sql("""
    SELECT QID, DERH_HHSEX, DERH_HSIZE
    FROM   census.households
    WHERE  ... AND ...
""").df()

---
# Part C — Counting and Summarising

Aggregates collapse many rows into one number. `GROUP BY` runs the aggregate once per group.

> **Text columns:** wrap numeric columns in `TRY_CAST(col AS DOUBLE)` before `AVG`/`MIN`/`MAX`.
> `TRY_CAST` returns `NULL` for values it cannot parse (e.g. household size `"10 +"`), which the
> aggregates then skip — no error.

## C1. Households by population group

In [ ]:
# TODO: GROUP BY DERH_HHPOP, COUNT(*) AS n_hh, ORDER BY n_hh DESC
con.sql("""
    SELECT DERH_HHPOP, COUNT(*) AS n_hh
    FROM   census.households
    GROUP BY ...
    ORDER BY ...
""").df()

## C2. Summarise household size

Average, minimum and maximum of `DERH_HSIZE` — remember the `TRY_CAST`.

In [ ]:
# TODO: AVG / MIN / MAX of TRY_CAST(DERH_HSIZE AS ...)
con.sql("""
    SELECT
        ROUND(AVG(TRY_CAST(DERH_HSIZE AS DOUBLE)), 2) AS mean_size,
        ...
    FROM census.households
""").df()

## C3. Households by tenure

Count households per `H03_TENURE`, largest first, top 5.

In [ ]:
# TODO: GROUP BY H03_TENURE, COUNT(*), ORDER BY count DESC, LIMIT 5
con.sql("""
    SELECT H03_TENURE, COUNT(*) AS n_hh
    FROM   census.households
    GROUP BY ...
    ORDER BY ...
    LIMIT 5
""").df()

---
# Part D — Joining Tables

The three tables share the household key `QID`. A `JOIN` combines them — the same idea as a
spreadsheet VLOOKUP, but in SQL.

## D1. Mean household size by settlement type

Join `households` to `geography` on `QID`, then average household size within each `Geo_type`.

In [ ]:
# TODO: JOIN households h and geography g ON h.QID = g.QID
#        GROUP BY g.Geo_type -> COUNT(*) AS n_hh, AVG household size
con.sql("""
    SELECT
        g.Geo_type,
        COUNT(*) AS n_hh,
        ROUND(AVG(TRY_CAST(h.DERH_HSIZE AS DOUBLE)), 2) AS mean_size
    FROM   census.households h
    JOIN   census.geography  g ON ...
    GROUP BY ...
    ORDER BY n_hh DESC
""").df()

## D2. Mean age of people by settlement type

Join `persons` to `geography` on `QID`, count people and average `P04_AGE` per `Geo_type`.

In [ ]:
# TODO: JOIN persons p and geography g ON p.QID = g.QID
#        GROUP BY g.Geo_type -> COUNT(*) AS n_people, AVG(TRY_CAST(p.P04_AGE AS DOUBLE))
con.sql("""
    SELECT
        g.Geo_type,
        COUNT(*) AS n_people,
        ROUND(AVG(TRY_CAST(p.P04_AGE AS DOUBLE)), 1) AS mean_age
    FROM   census.persons  p
    JOIN   census.geography g ON ...
    GROUP BY ...
    ORDER BY n_people DESC
""").df()

---
# Part E — From Database to Files (7.4)

The same DuckDB engine reads and writes **files**. `COPY (...query...) TO 'file'` streams a query
result straight to disk (no big pandas DataFrame needed), and DuckDB can then query that file
directly — the unifying idea of Lesson 7.4: **one SQL dialect over databases *and* files.**

## E1. Export a query result to Parquet and CSV

Write the D1 summary to Parquet, and the whole `geography` table to CSV.

In [ ]:
PARQUET_PATH = str(EXPORT_DIR / 'hh_size_by_geotype.parquet')
CSV_PATH     = str(EXPORT_DIR / 'geography.csv')

# TODO: COPY the D1 query result TO '{PARQUET_PATH}' (FORMAT PARQUET)
con.execute(f"""
    COPY (
        SELECT g.Geo_type, COUNT(*) AS n_hh,
               ROUND(AVG(TRY_CAST(h.DERH_HSIZE AS DOUBLE)), 2) AS mean_size
        FROM   census.households h
        JOIN   census.geography  g ON h.QID = g.QID
        GROUP BY g.Geo_type
    ) TO ... (FORMAT PARQUET)
""")

# TODO: COPY (SELECT * FROM census.geography) TO '{CSV_PATH}' (HEADER, DELIMITER ',')
con.execute(...)

print('wrote', PARQUET_PATH, 'and', CSV_PATH)

## E2. Query the files directly

No database, no `pd.read_*` — DuckDB reads the file straight from disk. The SQL is identical to what
you would write against a table.

In [ ]:
# TODO: query the Parquet file directly:  SELECT * FROM '{PARQUET_PATH}' ORDER BY mean_size DESC
display(con.sql(f"""
    SELECT *
    FROM   '{PARQUET_PATH}'
    ORDER BY ...
""").df())

# TODO: query the CSV file directly: count households per Geo_type
con.sql(f"""
    SELECT Geo_type, COUNT(*) AS n_hh
    FROM   '{CSV_PATH}'
    GROUP BY ...
    ORDER BY n_hh DESC
""").df()

---
# Summary — What you practised

Fill in the tool for each task.

| Task | Tool |
|---|---|
| Connect DuckDB to PostgreSQL | |
| Address a database table | |
| Filter on a text label | |
| Average a text-numeric column | |
| Combine tables on a key | |
| Save a result to a file | |
| Query a file with no import | |